In [1]:
import sys
print(sys.path) 

sys.path.insert(0, "/mnt/nas/pritish/CMUVERA_IR_ref")
sys.path.insert(0, "/raid/infolab/pritish/CMUVERA_IR_ref")
print(sys.path)

['', '/mnt/home/pritish/.local/lib/python3.11/site-packages', '/mnt/nas/pritish', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python311.zip', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/lib-dynload', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages', '/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages/setuptools/_vendor']
['/raid/infolab/pritish/CMUVERA_IR_ref', '/mnt/nas/pritish/CMUVERA_IR_ref', '', '/mnt/home/pritish/.local/lib/python3.11/site-packages', '/mnt/nas/pritish', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python311.zip', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/lib-dynload', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages', '/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT', '/mnt/nas/pritish/conda/.c

In [2]:
import torch
import os
import numpy as np
from omegaconf import OmegaConf
import time
import logging
from src.utils import set_seed,set_seed_from_checkpoint, load, save
from tqdm import tqdm
import random
from numpy.random import default_rng
import submodlib
import pickle

from src.dataloader import get_dataloader
from src.embedder import ColBERTEmbedder

from src.utils import partial_chamfer_sim_batched_with_rerank
import torch.multiprocessing as mp

# 1) Make sure the env var is set *inside* Python too
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

# 2) Turn on PyTorch’s deterministic‐only mode
torch.use_deterministic_algorithms(True)

from loguru import logger

/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages/beir/util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [3]:
%load_ext autoreload
%autoreload 1

%aimport src.endtoend
%aimport make_plots

In [4]:
config = {
    "k": 15,
    "method": "sml",
    "bucket_size": 0,
    "dataset_name": "scifact",
    "mv_type": 'colbertv2-plaid'
}

In [28]:
def make_config(config, optim="lazy"): # lazy, ltl, naive
    sys.argv = ["", f"k={config['k']}", f"method={config['method']}",
                f"baseline.bucket_size={config['bucket_size']}",
                f"data.dataset_name={config['dataset_name']}",
                f"embedder.mv_type={config['mv_type']}",
                f"submodlib.optimizer={optim}",
                f"embedder.mode={config.get('mode', 'mem')}"
    ]
    print(sys.argv)
    
    cli_conf = OmegaConf.from_cli()
    file_config = OmegaConf.load("configs/config.yaml")

    conf = OmegaConf.merge(file_config, cli_conf)

    return conf

In [6]:
os.chdir("CMUVERA_IR_ref")

In [7]:
conf = make_config(config)

['', 'k=15', 'method=sml', 'baseline.bucket_size=0', 'data.dataset_name=scifact', 'embedder.mv_type=colbertv2-plaid', 'submodlib.optimizer=lazy']


In [8]:
def get_method(config):
    name = config.method
    if name == "v0":
        return src.endtoend.GreedyBaseline_v0(config)
    elif name=="sml":
        return src.endtoend.GreedyBaseline_submodlib(config)
    else:
        raise ValueError(f"Unknown method: {name}")

In [9]:
retriever = get_method(conf)
retriever.run()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5183/5183 [00:00<00:00, 128999.22it/s]
/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()



#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: 0-dimensional biomaterials show inductive properties., 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  1014,  1011,  8789, 16012,  8585, 14482,  2015,  2265,
        27427, 14194,  6024,  5144,  1012,   102,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

./experiments/scifact/BERT/corpus/compressed_128/batch_0.0.pkl
0 1 loaded


/mnt/nas/pritish/CMUVERA_IR_ref/src/embedder.py:327: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cemb = torch.load(self.embedding_path(f"compressed_{self.config.emb_dim}",

0 2 loaded


Corpus: 1it [00:04,  4.77s/it]
[||||||||            ]40% [Iteration 6 of 15]██████████████████████████████████████████████████████████████████████████████████████████████████████| 300/300 [00:01<00:00, 163.12it/s]6% [Iteration 1 of 15]


(tensor([[ 331, 3300, 2331,  ..., 3878, 3878, 3878],
         [ 197, 2864,  903,  ..., 1072, 1072, 1072],
         [1098, 4141, 3834,  ..., 3711, 3711, 3711],
         ...,
         [1255,  213, 2238,  ..., 4944,  804,  804],
         [ 732, 2298, 3386,  ..., 3359, 3359, 3359],
         [4434, 2113, 1652,  ..., 4533, 4533, 4533]]),
 tensor([[ 5.3652,  7.8540,  9.3839,  ..., 11.4735, 11.4735, 11.4735],
         [ 9.8365, 11.2961, 11.6115,  ..., 12.7303, 12.7303, 12.7303],
         [ 7.0753,  7.6548,  7.8751,  ...,  8.9640,  8.9640,  8.9640],
         ...,
         [ 6.5853,  7.8433,  7.9079,  ..., 11.0056, 11.0056, 11.0056],
         [ 8.2209, 10.9956, 12.9946,  ..., 15.3333, 15.3333, 15.3333],
         [ 9.1629,  9.7375, 11.4678,  ..., 16.1609, 16.1609, 16.1609]]))

In [10]:
conf = make_config(config, optim="naive")

['', 'k=15', 'method=sml', 'baseline.bucket_size=0', 'data.dataset_name=scifact', 'embedder.mv_type=colbertv2-plaid', 'submodlib.optimizer=naive']


In [11]:
retriever = get_method(conf)
retriever.run()

Naive optimizer is not recommended, proceeding with caution
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5183/5183 [00:00<00:00, 148615.83it/s]



#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: 0-dimensional biomaterials show inductive properties., 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  1014,  1011,  8789, 16012,  8585, 14482,  2015,  2265,
        27427, 14194,  6024,  5144,  1012,   102,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

./experiments/scifact/BERT/corpus/compressed_128/batch_0.0.pkl
0 1 loaded
0 2 loaded


Corpus: 1it [00:04,  4.53s/it]
Query: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 300/300 [00:03<00:00, 97.33it/s]6% [Iteration 1 of 15]


(tensor([[ 331, 3300, 2331,  ..., 3878, 3878, 3878],
         [ 197, 2864,  903,  ..., 1072, 1072, 1072],
         [1098, 4141, 3834,  ..., 3711, 3711, 3711],
         ...,
         [1255,  213, 2238,  ..., 4944,  804,  804],
         [ 732, 2298, 3386,  ..., 3359, 3359, 3359],
         [4434, 2113, 1652,  ..., 4533, 4533, 4533]]),
 tensor([[ 5.3652,  7.8540,  9.3839,  ..., 11.4735, 11.4735, 11.4735],
         [ 9.8365, 11.2961, 11.6115,  ..., 12.7303, 12.7303, 12.7303],
         [ 7.0753,  7.6548,  7.8751,  ...,  8.9640,  8.9640,  8.9640],
         ...,
         [ 6.5853,  7.8433,  7.9079,  ..., 11.0056, 11.0056, 11.0056],
         [ 8.2209, 10.9956, 12.9946,  ..., 15.3333, 15.3333, 15.3333],
         [ 9.1629,  9.7375, 11.4678,  ..., 16.1609, 16.1609, 16.1609]]))

In [12]:
conf = make_config(config, optim="ltl")

['', 'k=15', 'method=sml', 'baseline.bucket_size=0', 'data.dataset_name=scifact', 'embedder.mv_type=colbertv2-plaid', 'submodlib.optimizer=ltl']


In [13]:
retriever = get_method(conf)
retriever.run()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5183/5183 [00:00<00:00, 147187.00it/s]



#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: 0-dimensional biomaterials show inductive properties., 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  1014,  1011,  8789, 16012,  8585, 14482,  2015,  2265,
        27427, 14194,  6024,  5144,  1012,   102,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

./experiments/scifact/BERT/corpus/compressed_128/batch_0.0.pkl
0 1 loaded
0 2 loaded


Corpus: 1it [00:04,  4.75s/it]
Query: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 300/300 [00:02<00:00, 128.71it/s]]6% [Iteration 1 of 15]


(tensor([[ 331, 2177, 1023,  ..., 1579, 1579, 1579],
         [4835, 2622,  197,  ..., 4828, 4828, 4828],
         [1142, 4972, 1432,  ..., 2870, 2870, 2870],
         ...,
         [4075,  745, 1882,  ..., 2165, 2165, 2165],
         [2298,  623, 3347,  ...,  227,  227,  227],
         [2762, 2113, 2944,  ..., 4809, 4809, 4809]]),
 tensor([[ 5.3652,  6.0558,  8.1750,  ...,  8.6809,  8.6809,  8.6809],
         [ 7.4117,  9.1415, 10.9016,  ..., 11.4747, 11.4747, 11.4747],
         [ 9.5569, 10.1521, 10.5212,  ..., 10.8131, 10.8131, 10.8131],
         ...,
         [ 6.4932,  9.2463, 11.7521,  ..., 11.7793, 11.7793, 11.7793],
         [ 9.6778, 11.3028, 12.1989,  ..., 14.8118, 14.8118, 14.8118],
         [ 8.2847,  9.2735, 10.3665,  ..., 14.5021, 14.5021, 14.5021]]))

In [14]:
# os.makedirs("plots/html",exist_ok=True)
# os.makedirs("plots/images",exist_ok=True)   
# # Load the data
# datasets = ["scifact"]
# b_sizes_aug = [10,25,50,100,200]
# k_values=[15]
# max_k = 15
# for data in datasets:
#     make_plots.plot_k_vs_metric(data,max_k,b_sizes_aug)
#     make_plots.plot_b_vs_metric(data,k_values,b_sizes_aug)

In [18]:
with open("./pickles/results/greedy_base_0_128_k15_scifact_bf.pkl", "rb") as f:
    results = pickle.load(f)

with open("./pickles/results/greedy_submodlib_NaiveGreedy_k15_scifact_bf_k15.pkl", "rb") as f:
    results_naive = pickle.load(f)

with open("./pickles/results/greedy_submodlib_LazyGreedy_k15_scifact_bf_k15.pkl", "rb") as f:
    results_lazy = pickle.load(f)

with open("./pickles/results/greedy_submodlib_LazierThanLazyGreedy_k15_scifact_bf_k15.pkl", "rb") as f:
    results_ltl = pickle.load(f)

In [23]:
results_naive[0][0], results_naive[1][0]

(tensor([ 331, 3300, 2331,  496, 3816, 3878, 3878, 3878]),
 tensor([ 5.3652,  7.8540,  9.3839, 11.3462, 11.3462, 11.4735, 11.4735, 11.4735]))

In [24]:
results_lazy[0][0], results_lazy[1][0]

(tensor([ 331, 3300, 2331,  496, 3816, 3878, 3878, 3878]),
 tensor([ 5.3652,  7.8540,  9.3839, 11.3462, 11.3462, 11.4735, 11.4735, 11.4735]))

In [25]:
results_ltl[0][0], results_ltl[1][0]

(tensor([ 331, 2177, 1023, 4497, 1579, 1579, 1579, 1579, 1579]),
 tensor([5.3652, 6.0558, 8.1750, 8.6593, 8.6809, 8.6809, 8.6809, 8.6809, 8.6809]))

In [26]:
results_naive[0][-1], results_naive[1][-1]

(tensor([4434, 2113, 1652, 4994,   92, 4533, 4533, 4533]),
 tensor([ 9.1629,  9.7375, 11.4678, 11.7999, 16.1609, 16.1609, 16.1609, 16.1609]))

In [19]:
results[0]

[(2716, 11.888933181762695),
 (1858, 15.326838493347168),
 (2660, 17.4311466217041),
 (2022, 18.882389068603516),
 (1853, 19.841398239135742),
 (190, 20.44447898864746),
 (1321, 20.73671531677246),
 (916, 20.99953842163086),
 (4404, 21.165863037109375),
 (1421, 21.295223236083984),
 (1632, 21.38498306274414),
 (2967, 21.44413185119629),
 (1864, 21.495826721191406),
 (4658, 21.539615631103516),
 (2495, 21.570064544677734)]

In [44]:
import copy
config_disk_mode = copy.deepcopy(config)
config_disk_mode["mode"] = "disk"
conf = make_config(config_disk_mode, optim="naive")

['', 'k=15', 'method=sml', 'baseline.bucket_size=0', 'data.dataset_name=scifact', 'embedder.mv_type=colbertv2-plaid', 'submodlib.optimizer=naive', 'embedder.mode=disk']


In [50]:
retriever = get_method(conf)
retriever.run()

Naive optimizer is not recommended, proceeding with caution
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5183/5183 [00:00<00:00, 146706.24it/s]



#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: 0-dimensional biomaterials show inductive properties., 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  1014,  1011,  8789, 16012,  8585, 14482,  2015,  2265,
        27427, 14194,  6024,  5144,  1012,   102,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

./experiments/scifact/BERT/corpus/compressed_128/batch_0.0.pkl
DISK MODE


Corpus: 0it [00:00, ?it/s]

I modified this line


Corpus: 1it [00:01,  1.04s/it]

...

I modified this line


Corpus: 2it [00:01,  1.24it/s]
[||||||||            ]40% [Iteration 6 of 15]██████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 101.61it/s]]]


Batch 0 Minibatch 0
Batch 0 Minibatch 1


Processing queries: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 300/300 [00:00<00:00, 6850.56it/s]


(tensor([[ 331, 3300, 2331,  ..., 3878, 3878, 3878],
         [ 197, 2864,  903,  ..., 1072, 1072, 1072],
         [1098, 4141, 3834,  ..., 3711, 3711, 3711],
         ...,
         [1255,  213, 2238,  ..., 4944,  804,  804],
         [ 732, 2298, 3386,  ..., 3359, 3359, 3359],
         [4434, 2113, 1652,  ..., 4533, 4533, 4533]]),
 tensor([[ 5.3652,  7.8540,  9.3839,  ..., 11.4735, 11.4735, 11.4735],
         [ 9.8365, 11.2961, 11.6115,  ..., 12.7303, 12.7303, 12.7303],
         [ 7.0753,  7.6548,  7.8751,  ...,  8.9640,  8.9640,  8.9640],
         ...,
         [ 6.5853,  7.8433,  7.9079,  ..., 11.0056, 11.0056, 11.0056],
         [ 8.2209, 10.9956, 12.9946,  ..., 15.3333, 15.3333, 15.3333],
         [ 9.1629,  9.7375, 11.4678,  ..., 16.1609, 16.1609, 16.1609]]))

In [51]:
with open("./pickles/results/greedy_submodlib_NaiveGreedy_k15_scifact_bf_k15.pkl", "rb") as f:
    results_naive_disk_mode = pickle.load(f)

In [52]:
results_naive_disk_mode[0][0], results_naive_disk_mode[1][0]

(tensor([ 331, 3300, 2331,  496, 3816, 3878, 3878, 3878]),
 tensor([ 5.3652,  7.8540,  9.3839, 11.3462, 11.3462, 11.4735, 11.4735, 11.4735]))

In [35]:
for i in range(len(results_naive_disk_mode[0])):
    # Assert that results_naive and results_naive_disk_mode are the same
    assert np.allclose(results_naive_disk_mode[0][i], results_naive[0][i]), f"Mismatch at index {i}"
    assert np.allclose(results_naive_disk_mode[1][i], results_naive[1][i]), f"Mismatch at index {i}"

In [47]:
results_naive_disk_mode[0][0], results_naive[0][0], results[0]

(tensor([ 331, 3300, 2331,  496, 3816, 3878, 3878, 3878]),
 tensor([ 331, 3300, 2331,  496, 3816, 3878, 3878, 3878]),
 [(2716, 11.888933181762695),
  (1858, 15.326838493347168),
  (2660, 17.4311466217041),
  (2022, 18.882389068603516),
  (1853, 19.841398239135742),
  (190, 20.44447898864746),
  (1321, 20.73671531677246),
  (916, 20.99953842163086),
  (4404, 21.165863037109375),
  (1421, 21.295223236083984),
  (1632, 21.38498306274414),
  (2967, 21.44413185119629),
  (1864, 21.495826721191406),
  (4658, 21.539615631103516),
  (2495, 21.570064544677734)])

In [38]:
results_naive[0].shape

torch.Size([300, 8])

In [54]:
retriever.embedder.qembs.shape

torch.Size([300, 32, 128])